In [ ]:
from PIL import Image
import numpy as np
import time
import math

def message_to_bits(message):
    return ''.join(format(ord(char), '08b') for char in message)

def bits_to_message(bits):
    chars = [bits[i:i+8] for i in range(0, len(bits), 8)]
    return ''.join(chr(int(char, 2)) for char in chars)

def embed_lsb_jpg(input_jpg, secret_message, output_png):
    # Open image and convert to RGB (necessary for consistent pixel structure)
    image = Image.open(input_jpg).convert("RGB")
    pixels = np.array(image)

    # Flatten the pixel array to easily access individual color components
    flat_pixels = pixels.flatten()

    # Convert secret message to a binary string
    message_bits = message_to_bits(secret_message)
    message_length = len(message_bits)

    # Check if the message can fit in the image
    if message_length > len(flat_pixels):
        raise ValueError("Message is too large for this image.")

    start_time = time.time()

    # Embed each bit into the LSB of the corresponding pixel component
    for i, bit in enumerate(message_bits):
        # Clear LSB (AND with 0xFE which is 11111110) and insert secret bit
        flat_pixels[i] = (flat_pixels[i] & 0xFE) | int(bit)

    embedding_time = (time.time() - start_time) * 1000

    # Reshape the flattened array back to the original image dimensions
    stego_pixels = flat_pixels.reshape(pixels.shape)

    # Create a new PIL image from the modified NumPy array
    stego_image = Image.fromarray(stego_pixels.astype(np.uint8), "RGB")

    # Save as PNG (lossless format is crucial for LSB steganography)
    stego_image.save(output_png, "PNG")

    return pixels, stego_pixels, message_length, embedding_time

def extract_lsb_jpg(stego_png, message_length):
    # Open the steganographic PNG image
    image = Image.open(stego_png).convert("RGB")
    pixels = np.array(image).flatten()

    # Extract LSB from each pixel component up to the message length
    extracted_bits = ''.join(str(pixels[i] & 1) for i in range(message_length))

    # Convert the extracted bits back to the original message
    return bits_to_message(extracted_bits)

def calculate_mse(original, stego):
    return np.mean((original.astype(float) - stego.astype(float)) ** 2)

def calculate_psnr(mse):
    if mse == 0:
        return float("inf")

    return 10 * math.log10((255 ** 2) / mse)


# Simulated IoT sensor message
secret_message = "DEVICE_ID=Sensor_12;TEMP=27.4;STATUS=OK;AUTH=9F3A21"

# Run embedding
# Change output file name to .png
original, stego, payload_bits, embedding_time = embed_lsb_jpg(
    "cover_image.jpg",
    secret_message,
    "stego_image.png" # Save as PNG
)

# Extract hidden message
# Change input file name to .png
extracted_message = extract_lsb_jpg(
    "stego_image.png", # Load from PNG
    payload_bits
)

# Quality metrics
mse = calculate_mse(original, stego)
psnr = calculate_psnr(mse)

# Results
print("Original message:", secret_message)
print("Extracted message:", extracted_message)
print("Payload size:", payload_bits, "bits")
print("Embedding time:", round(embedding_time, 4), "ms")
print("MSE:", mse)
print("PSNR:", psnr)
print("Extraction successful:", secret_message == extracted_message)

Original message: DEVICE_ID=Sensor_12;TEMP=27.4;STATUS=OK;AUTH=9F3A21
Extracted message: ¨ÜÇãþ ÿðÿþ?ÿÿÿþp	Û#;ù#ÍSó¼¶~8 Ç8p8
Payload size: 408 bits
Embedding time: 0.257 ms
MSE: 2.389253880436453e-05
PSNR: 94.34818060839929
Extraction successful: False


/tmp/ipykernel_1474/3183036522.py:35: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  stego_image = Image.fromarray(stego_pixels.astype(np.uint8), "RGB")


# New Section

# New Section